# สร้างรายการงบสุดท้าย

ไฟล์นี้จะสร้างรายการงบที่เป็นไฟล์สุดท้ายที่เราต้องการ โดยทำจากต้นไม้ในขั้นตอนก่อนหน้า

ไฟล์นี้จะ**เพิ่ม**แผ่นงานใน spreadseet `BUDGET_LIST_GSPREADSHEET_KEY` ด้วยชื่อ `ปี-เดือน-วัน`

## สิ่งที่ไฟล์นี้ใช้

1. ไฟล์นี้จะขอสิทธิ์การเขียนและอ่านไฟล์ใน gdrive
1. กำหนด key ของ spread sheet ให้ถูกต้อง

## วิธีใช้

1. กด `Runtime` > `Run all`

In [ ]:
BUDGET_YEAR = '2570'
BUDGET_TREE_GSPREADSHEET_KEY = '1eNd8NmGAn3xGlXpLztPVyUXZOQW98I4ZR_XISeBK-88'
BUDGET_LIST_GSPREADSHEET_KEY = '12eHF1-cKS0y1q0eqsHrxHXBYTUtGrx3L7J3Vvjg2qE0'

In [ ]:
!rm -rf wv-th-bdgt example test thbud && git clone https://github.com/napatswift/wv-th-bdgt && mv wv-th-bdgt/* . && pip install -qq -r requirements.txt

Cloning into 'wv-th-bdgt'...
remote: Enumerating objects: 565, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 565 (delta 72), reused 113 (delta 41), pack-reused 409 (from 1)
Receiving objects: 100% (565/565), 4.44 MiB | 10.85 MiB/s, done.
Resolving deltas: 100% (301/301), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 3.0 MB/s eta 0:00:00


In [ ]:
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
from thbud.model import BudgetItem
from thbud.build_csv import extract_budget_item_name
from thbud import build_csv
import pandas as pd
import datetime
import numpy as np
import re

In [ ]:
def np_encode(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj

In [ ]:
spreadsheet = gc.open_by_key(BUDGET_TREE_GSPREADSHEET_KEY)

In [ ]:
def read_worksheet(worksheet):
    values = worksheet.get_all_values()
    columns = values[0]
    row_list = []
    for row_idx in range(1, len(values)):
        row = values[row_idx]
        row_data = dict(row_index=row_idx)
        row_list.append(row_data)
        for colval, colname in zip(row, columns):
            value = colval

            if colname == 'amount':
                try:
                    value = float(value.replace(',', ''))
                except ValueError as e:
                    raise ValueError(f'{e}: {row_data}')

            if colname == 'page':
                try:
                    value = None if value == '' else int(value)
                except ValueError as e:
                    raise ValueError(f'{e}: {row_data}')

            if colname.startswith('name_'):
                if not value:
                    continue
                else:
                    value = extract_budget_item_name(value)
                    value = re.sub(r'\s+', ' ', value).strip()

            if colname.startswith('fiscal_year'):

                if not value:
                    continue
                try:
                    value = int(value)
                except ValueError as e:
                    raise ValueError(f'{e}: {row_data}')


            row_data[colname] = value
    return row_list



In [ ]:
budget_rows = []
for sheet in spreadsheet.worksheets():
    if (
        sheet.title == 'README'
        or sheet.title == 'CHECKLIST'
        or sheet.title == 'PROGRESS'
        or sheet.title == 'Crosscheck'
        or sheet.title.startswith('(')
        or sheet.title.startswith('[')
        or not sheet.title.startswith('DONE')
    ): continue

    print('Working on', repr(sheet.title), 'sheet')
    root = BudgetItem.build_tree_by_rows(read_worksheet(sheet))
    budget_rows += build_csv(root)

df = pd.DataFrame(budget_rows)
df.loc[df['FISCAL_YEAR'].isna(), 'FISCAL_YEAR'] = BUDGET_YEAR
df['FISCAL_YEAR'] = df.FISCAL_YEAR.astype(int) - 543

Working on 'DONEหน่วยงานขององค์กรอิสระและองค์กรอัยการ' sheet
Working on 'DONEกระทรวงคมนาคม' sheet
Working on 'DONEกระทรวงทรัพยากรธรรมชาติและสิ่งแวดล้อม' sheet
Working on 'DONEรัฐวิสาหกิจ' sheet
Working on 'DONEสภากาชาดไทย' sheet
Working on 'DONEกระทรวงพลังงาน' sheet
Working on 'DONEหน่วยงานของศาล' sheet
Working on 'DONEส่วนราชการในพระองค์' sheet
Working on 'DONEกระทรวงแรงงาน' sheet
Working on 'DONEจังหวัดและกลุ่มจังหวัด' sheet
Working on 'DONEหน่วยงานของรัฐสภา' sheet
Working on 'DONEกระทรวงการคลัง' sheet
Working on 'DONEกระทรวงพาณิชย์' sheet
Working on 'DONEกระทรวงการพัฒนาสังคมและความมั่นคงของมนุษย์' sheet
Working on 'DONEงบกลาง' sheet
Working on 'DONEกระทรวงเกษตรและสหกรณ์' sheet
Working on 'DONEกระทรวงอุตสาหกรรม' sheet
Working on 'DONEกระทรวงการต่างประเทศ' sheet
Working on 'DONEกระทรวงการอุดมศึกษาฯ' sheet
Working on 'DONEกระทรวงวัฒนธรรม' sheet
Working on 'DONEกระทรวงสาธารณสุข' sheet
Working on 'DONEกระทรวงดิจิทัลเพื่อเศรษฐกิจและสังคม' sheet
Working on 'DONEสำนักนายกรัฐมนตรี' sheet
Wor

/tmp/ipykernel_829/3617997036.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2570' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df['FISCAL_YEAR'].isna(), 'FISCAL_YEAR'] = BUDGET_YEAR


In [ ]:
df[df.FISCAL_YEAR==2025].groupby('MINISTRY')['AMOUNT'].sum()

,AMOUNT
MINISTRY,
กระทรวงกลาโหม,1.069940e+10
กระทรวงการคลัง,5.027740e+09
กระทรวงการต่างประเทศ,1.554496e+08
กระทรวงการท่องเที่ยวและกีฬา,1.492582e+08
กระทรวงการพัฒนาสังคมและความมั่นคงของมนุษย์,2.566200e+07
กระทรวงการอุดมศึกษาฯ,2.878481e+09
กระทรวงคมนาคม,4.228771e+10
กระทรวงดิจิทัลเพื่อเศรษฐกิจและสังคม,6.532222e+08
กระทรวงทรัพยากรธรรมชาติและสิ่งแวดล้อม,1.155186e+09


In [ ]:
bl_spreadsheet = gc.open_by_key(BUDGET_LIST_GSPREADSHEET_KEY)

In [ ]:
column_names = ['REF_DOC', 'REF_PAGE_NO', 'MINISTRY', 'BUDGETARY_UNIT',
        # 'STRATEGY', 'MOTHER_PLAN',
        'BUDGET_PLAN', 'CROSS_FUNC?',
        'OUTPUT', 'PROJECT', ]

column_names += sorted([c for c in df.columns if c.startswith('CATEGORY_LV')], key=lambda x: int(x[11:]))

column_names += ['ITEM_DESCRIPTION', 'AMOUNT', 'FISCAL_YEAR', 'OBLIGED?']

df = df[column_names]

In [ ]:
sheet_title = datetime.date.today().isoformat() + '#'
worksheet = bl_spreadsheet.add_worksheet(sheet_title, len(df)+1, len(df.columns))
cells = worksheet.range(1, 1, len(df)+1, len(df.columns))

for cell in cells:
    if cell.row == 1:
        new_val = df.columns[cell.col-1]
    else:
        new_val = '' if pd.isna(df.iloc[cell.row-2, cell.col-1]) else df.iloc[cell.row-2, cell.col-1]

    cell.value = np_encode(new_val)

worksheet.update_cells(cells)